In [1]:
!pip install -q transformers peft bitsandbytes accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00


In [2]:
import os
import time
import torch
import shutil
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

GPU: Tesla T4
VRAM: 15.64 GB


In [3]:
from google.colab import files
uploaded = files.upload()

Saving tokenizer.json to tokenizer.json
Saving adapter_model.safetensors to adapter_model.safetensors
Saving adapter_config.json to adapter_config.json


In [4]:
os.makedirs("/content/adapters", exist_ok=True)
for f in ["adapter_model.safetensors", "adapter_config.json", "tokenizer.json"]:
    if os.path.exists(f"/content/{f}"):
        shutil.move(f"/content/{f}", f"/content/adapters/{f}")
print(os.listdir("/content/adapters"))

['adapter_model.safetensors', 'adapter_config.json', 'tokenizer.json']


In [5]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer  = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype      = torch.float16,
    device_map = "auto"
)
print("base model loaded")
print("memory:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

base model loaded
memory: 2.2 GB


In [6]:
model = PeftModel.from_pretrained(base_model, "/content/adapters")
model = model.merge_and_unload()
print("merged")
print("memory:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

merged
memory: 2.21 GB


In [7]:
os.makedirs("/content/quantized/model-fp16", exist_ok=True)
model.save_pretrained("/content/quantized/model-fp16")
tokenizer.save_pretrained("/content/quantized/model-fp16")
print("FP16 saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

FP16 saved


In [8]:
bnb_int8   = BitsAndBytesConfig(load_in_8bit=True)
model_int8 = AutoModelForCausalLM.from_pretrained(
    "/content/quantized/model-fp16",
    quantization_config = bnb_int8,
    device_map          = "auto"
)
os.makedirs("/content/quantized/model-int8", exist_ok=True)
model_int8.save_pretrained("/content/quantized/model-int8")
tokenizer.save_pretrained("/content/quantized/model-int8")
print("INT8 saved")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT8 saved


In [9]:
bnb_int4   = BitsAndBytesConfig(
    load_in_4bit           = True,
    bnb_4bit_compute_dtype = torch.float16,
    bnb_4bit_quant_type    = "nf4"
)
model_int4 = AutoModelForCausalLM.from_pretrained(
    "/content/quantized/model-fp16",
    quantization_config = bnb_int4,
    device_map          = "auto"
)
os.makedirs("/content/quantized/model-int4", exist_ok=True)
model_int4.save_pretrained("/content/quantized/model-int4")
tokenizer.save_pretrained("/content/quantized/model-int4")
print("INT4 saved")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT4 saved


In [10]:
!git clone https://github.com/ggerganov/llama.cpp -q
!pip install -q -r /content/llama.cpp/requirements.txt

from huggingface_hub import hf_hub_download
hf_hub_download(
    repo_id   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    filename  = "tokenizer.model",
    local_dir = "/content/quantized/model-fp16"
)

!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/quantized/model-fp16 \
    --outfile /content/quantized/model.gguf \
    --outtype q8_0

print("GGUF size:", round(os.path.getsize("/content/quantized/model.gguf") / 1e9, 2), "GB")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 47.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 111.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.6/178.6 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.6/343.6 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.9 MB/s eta 0:00:00
  

In [ ]:
from google.colab import files
files.download("/content/quantized/model.gguf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
import os
import time
import torch
import subprocess
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

BASE_PATH = "/content/quantized"

PROMPT = "Write a Python function to check if a number is prime."
MAX_NEW_TOKENS = 128


MODELS = {
    "FP16": {
        "path": f"{BASE_PATH}/model-fp16",
        "quant_config": None,
        "torch_dtype": torch.float16
    },
    "INT8": {
        "path": f"{BASE_PATH}/model-int8",
        "quant_config": BitsAndBytesConfig(
            load_in_8bit=True
        ),
        "torch_dtype": None
    },
    "INT4": {
        "path": f"{BASE_PATH}/model-int4",
        "quant_config": BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        ),
        "torch_dtype": None
    }
}


def get_model_size(path):
    result = subprocess.check_output(["du", "-sh", path]).decode()
    return result.split()[0]


def benchmark_model(name, cfg):
    print(f"\n--- Benchmarking {name} ---")

    tokenizer = AutoTokenizer.from_pretrained(
        cfg["path"],
        local_files_only=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        cfg["path"],
        device_map="auto",
        local_files_only=True,
        torch_dtype=cfg["torch_dtype"],
        quantization_config=cfg["quant_config"]
    )

    inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)

    torch.cuda.synchronize()
    start = time.time()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS
        )

    torch.cuda.synchronize()
    end = time.time()

    tokens_generated = output.shape[-1] - inputs["input_ids"].shape[-1]
    tokens_per_sec = tokens_generated / (end - start)

    size = get_model_size(cfg["path"])

    del model
    torch.cuda.empty_cache()

    return {
        "format": name,
        "size": size,
        "tokens_per_sec": round(tokens_per_sec, 2)
    }


results = []

for name, cfg in MODELS.items():
    res = benchmark_model(name, cfg)
    results.append(res)


print("\n===== BENCHMARK RESULTS =====")
for r in results:
    print(
        f"{r['format']:>5} | Size: {r['size']:>6} | Speed: {r['tokens_per_sec']} tokens/sec"
    )



--- Benchmarking FP16 ---


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


--- Benchmarking INT8 ---


/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  def supports_quant_method(quantization_config_dict):


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


--- Benchmarking INT4 ---


/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  def supports_quant_method(quantization_config_dict):


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


===== BENCHMARK RESULTS =====
 FP16 | Size:   2.1G | Speed: 25.59 tokens/sec
 INT8 | Size:   1.2G | Speed: 9.73 tokens/sec
 INT4 | Size:   774M | Speed: 24.08 tokens/sec


In [12]:
prompts = [
    "Write a Python function to check if a number is prime.",
    "Explain the difference between a list and tuple in Python.",
    "Write a SQL query to find duplicate records in a table."
]

def test_quality(model_path, label, bnb_config=None):
    tok = AutoTokenizer.from_pretrained(model_path)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_path,
        quantization_config = bnb_config,
        device_map          = "auto",
        dtype               = torch.float16
    )
    print(f"\n{'='*10} {label} {'='*10}")
    for prompt in prompts:
        inputs = tok(prompt, return_tensors="pt").to(mdl.device)
        output = mdl.generate(**inputs, max_new_tokens=120)
        result = tok.decode(output[0], skip_special_tokens=True)
        print(f"\nPrompt : {prompt}")
        print(f"Output : {result}")
    del mdl
    torch.cuda.empty_cache()

# FP16
test_quality("/content/quantized/model-fp16", "FP16")

# INT8
test_quality("/content/quantized/model-int8", "INT8",
    BitsAndBytesConfig(load_in_8bit=True))

# INT4
test_quality("/content/quantized/model-int4", "INT4",
    BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4"))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


========== FP16 ==========

Prompt : Write a Python function to check if a number is prime.
Output : Write a Python function to check if a number is prime. The function should return True if the number is prime and False otherwise. The function should take a single argument, the number to be checked.

# Function to check if a number is prime
def is_prime(num):
    # Check if the number is less than 2
    if num < 2:
        return False
    
    # Check if the number is divisible by any other number
    for i in range(2, num):
        if num % i == 0:
            return False
    
    return True



Prompt : Explain the difference between a list and tuple in Python.
Output : Explain the difference between a list and tuple in Python.
Explain the difference between a list and tuple in Python.

A: A list is a mutable sequence of objects. It can be created with the list() function, and can be assigned to a variable. It can also be iterated over using the for loop.
A tuple is a fixed-lengt

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  def supports_quant_method(quantization_config_dict):


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


========== INT8 ==========

Prompt : Write a Python function to check if a number is prime.
Output : Write a Python function to check if a number is prime. The function should return True if the number is prime and False otherwise. The function should take a single argument, the number to be checked.

# Function to check if a number is prime
def is_prime(num):
    if num <= 1:
        return False
    for i in range(2, num):
        if num % i == 0:
            return False
    return True

# Example usage
print(is_prime(10)) # Output: True
print(is_prime(12)) # Output:

Prompt : Explain the difference between a list and tuple in Python.
Output : Explain the difference between a list and tuple in Python.
Explain the difference between a list and tuple in Python.

A: A list is a mutable sequence of objects, while a tuple is a fixed-length sequence of objects. A list can be modified in-place, while a tuple cannot. A list can be indexed, while a tuple cannot. A list can be sliced, while 

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


========== INT4 ==========

Prompt : Write a Python function to check if a number is prime.
Output : Write a Python function to check if a number is prime. The function should take a number and return True if the number is prime, False otherwise. The function should also handle negative numbers.

# Define the function
def is_prime(num):
    # Check if the number is less than 2
    if num < 2:
        return False
    # Check if the number is divisible by 2
    for i in range(2, int(num/2)+1):
        if num % i == 0:
            return False
    return True

# Test the function
print

Prompt : Explain the difference between a list and tuple in Python.
Output : Explain the difference between a list and tuple in Python.
Explain the difference between a list and tuple in Python.

A: A list is a mutable sequence of objects. It can be used to store and manipulate data. It can be used to store and manipulate objects of any type. A tuple is a fixed-length sequence of objects. It can be used 